In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

In [14]:
df = pd.read_parquet("merged_features.parquet")

In [15]:
df

,total_amount,mean_amount,std_amount,min_amount,max_amount,median_amount,transaction_count,credit_sum,debit_sum,credit_debit_ratio,...,fft_high_freq_power_cat31.0,fft_power_ratio_cat31.0,fft_dominant_freq_cat33.0,fft_dominant_power_cat33.0,fft_spectral_entropy_cat33.0,fft_low_freq_power_cat33.0,fft_high_freq_power_cat33.0,fft_power_ratio_cat33.0,dataset_type,FPF_TARGET
masked_consumer_id,,,,,,,,,,,,,,,,,,,,,
C01100001,-8111.42,-3.051701,645.540740,-8000.00,8483.00,-25.380,2658,276864.26,284975.68,0.971536,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100002,-9490.73,-3.813070,910.144154,-19150.00,20000.00,-25.290,2489,285876.77,295367.50,0.967868,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100003,2137.77,3.224389,748.056684,-4310.28,4409.40,-21.350,663,84178.75,82040.98,1.026057,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100004,-29837.87,-27.525710,414.375001,-1800.00,1748.00,-27.555,1084,97426.62,127264.49,0.765544,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100005,-9756.37,-4.044930,1314.772339,-6987.07,39915.16,-25.080,2412,404922.85,414679.22,0.976472,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C01104996,38478.08,12.384319,1634.482084,-37648.00,44100.00,-26.530,3107,470767.11,432289.03,1.089010,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01104997,591.59,0.512199,4459.475583,-56256.44,134384.01,-19.990,1155,217927.74,217336.15,1.002722,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01104998,-12120.38,-3.429649,394.044758,-1953.37,12710.00,-17.715,3534,136083.82,148204.20,0.918218,...,0.0,0.0,0.210526,16857.338694,1.894851,0.0,21227.966021,0.0,real,0.0


In [16]:
# Group by customer_category and FPF_TARGET, then count occurrences
category_target_counts = (
    df.groupby(["dataset_type", "FPF_TARGET"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0.0: "count_0", 1.0: "count_1"})
)
category_target_counts

FPF_TARGET,count_0,count_1
dataset_type,,
real,3942,58


In [17]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score, accuracy_score

# If masked_consumer_id is the index, make it a column:
df = df.reset_index().rename(columns={'index':'masked_consumer_id'})
# Derive customer_category from the ID prefix (assumes IDs start like 'C01_…')
df['customer_category'] = df['masked_consumer_id'].str.extract(r'^(C\d{2})')

# 1) Define customer categories and model pipelines
categories = ["C01"]

# model_pipelines = {
#     "XGBoost": Pipeline([
#         ('scaler', StandardScaler()),
#         ('model', XGBClassifier(
#             n_estimators=100,
#             max_depth=5,
#             learning_rate=0.01,
#             subsample=0.85,
#             colsample_bytree=0.85,
#             scale_pos_weight=1,
#             eval_metric='logloss',
#             random_state=42
#         ))
#     ])
# }
model_pipelines = {
    "XGBoost": Pipeline([
        ('scaler', StandardScaler()),
        ('model', XGBClassifier(
            n_estimators=100,
            max_depth=20,
            learning_rate=0.01,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=1,
            eval_metric='logloss',
            random_state=42
        ))
    ])
}

results = {}

for cat in categories:
    print(f"\n=== Category: {cat} ===")
    df_cat = df[df["customer_category"] == cat].copy()
    if df_cat.shape[0] < 10:
        print("  skipping: too few samples")
        continue

    # 2) split real vs simulation
    mask_sim = df_cat['masked_consumer_id'].str.contains(r'_simulation_')
    df_real  = df_cat.loc[~mask_sim].reset_index(drop=True)
    df_sim   = df_cat.loc[ mask_sim].reset_index(drop=True)

    # 3) real-only stratified split
    Xr = df_real.drop(columns=[
        'masked_consumer_id',
        'customer_category',
        'FPF_TARGET',
        'dataset_type'
    ])
    yr = df_real['FPF_TARGET']
    feat_names = Xr.columns.tolist()

    Xr_tr, Xr_val, yr_tr, yr_val = train_test_split(
        Xr, yr,
        test_size=0.2,
        stratify=yr,
        random_state=42
    )

    # 4) bring in only those sims whose parent is in the training real IDs
    df_sim['parent_id'] = df_sim['masked_consumer_id'].str.replace(r'_simulation_\d+$','', regex=True)
    train_ids = set(df_real.loc[Xr_tr.index, 'masked_consumer_id'])
    df_sim_tr = df_sim.loc[df_sim['parent_id'].isin(train_ids)].drop(columns=['parent_id'])
    X_sim = df_sim_tr[feat_names]
    y_sim = df_sim_tr['FPF_TARGET']

    # 5) assemble final train/val sets
    X_train = pd.concat([Xr_tr, X_sim], axis=0)
    y_train = pd.concat([yr_tr, y_sim], axis=0)
    X_val, y_val = Xr_val, yr_val

    # 6) fit L2 logistic to select top 50 features
    l2 = LogisticRegression(penalty='l2',
                            max_iter=2000, random_state=42)
    l2.fit(X_train, y_train)
    coefs = np.abs(l2.coef_).ravel()
    top_idx = np.argsort(coefs)[::-1][:250]
    selected_feats = [feat_names[i] for i in top_idx]
    print(f"Selected top 250 features for {cat}: {selected_feats}")

    # 7) subset to top 250
    X_train_sel = X_train.iloc[:, top_idx]
    X_val_sel   = X_val.iloc[:,   top_idx]

    # 8) train and evaluate each pipeline
    for name, pipe in model_pipelines.items():
        print(f"→ Training {name} on top 250 features")
        pipe.fit(X_train_sel, y_train)
        p_tr = pipe.predict_proba(X_train_sel)[:,1]
        p_val = pipe.predict_proba(X_val_sel)[:,1]
        train_auc = roc_auc_score(y_train, p_tr)
        val_auc   = roc_auc_score(y_val,   p_val)
        print(f"   Train AUC={train_auc:.3f}, Val AUC={val_auc:.3f}")

        # Optional: store results
        results.setdefault(cat, {})[name] = {'train_auc': train_auc, 'val_auc': val_auc}


=== Category: C01 ===


/opt/miniconda3/envs/erdos_spring_2025/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Selected top 250 features for C01: ['fft_dominant_power_cat12.0', 'fft_low_freq_power_cat26.0', 'fft_high_freq_power_cat17.0', 'fft_dominant_power_cat11.0', 'fft_high_freq_power_cat23.0', 'fft_high_freq_power_cat11.0', 'fft_dominant_power_cat16.0', 'fft_low_freq_power_cat14.0', 'fft_high_freq_power_cat34.0', 'fft_high_freq_power_cat6.0', 'fft_high_freq_power_cat24.0', 'fft_dominant_power_cat26.0', 'fft_high_freq_power_cat32.0', 'fft_high_freq_power_cat20.0', 'fft_dominant_power_cat14.0', 'fft_high_freq_power_cat9.0', 'fft_dominant_power_cat32.0', 'fft_dominant_power_cat17.0', 'fft_dominant_power_cat34.0', 'fft_dominant_power_cat6.0', 'fft_high_freq_power_cat27.0', 'fft_dominant_power_cat13.0', 'fft_low_freq_power_cat16.0', 'fft_low_freq_power_cat11.0', 'fft_low_freq_power_cat8.0', 'fft_low_freq_power_cat21.0', 'fft_dominant_power_cat8.0', 'fft_high_freq_power_cat30.0', 'fft_high_freq_power_cat18.0', 'fft_low_freq_power_cat17.0', 'fft_dominant_power_cat20.0', 'fft_dominant_power_cat24.0